# 19 - Click-and-track ground truth (every frame)

One click per trace: the object in a frame where it is clearly visible (~3 m).
The SAM3 video tracker follows it forward until it leaves the view and
backward to where it first appears - a box (or "not visible") for **every**
frame within 8 m. You check a filmstrip per trace and fix drift with more
clicks or "not visible after/before frame X". Output is dense ground truth in
the gold-CSV format `scripts/rate_autolabel.py` reads.

Default scope: the 24 held-out test traces (8,729 frames). `ALL_TRACES = True`
does all 72 (dense training labels too).

Everything saves to `<artifacts>/gold/clicktrack_v1/` and resumes: clicks are
asked once, a trace is re-tracked only when its clicks or fixes change.
**GPU runtime** and Hugging Face access to `facebook/sam3` (as notebook 02).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import hashlib
import json as _json
import shutil
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from vision_uss_research.click_track import (apply_cutoffs, mask_to_box,
                                             merge_by_nearest_click,
                                             stop_after_loss, to_gold_rows)
from vision_uss_research.cuv_finetune import final_approach
from vision_uss_research.labeling.click_ui import ask_click_colab, ask_click_local
from vision_uss_research.silver_gt import select_test_items

cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / "cuv_finetune_v1.yaml").read_text())
ALL_TRACES = False        # True: all 72 traces, not only the 24 test traces
CLICK_DIST_M = 3.0        # click frame: the one closest to this distance
BOX_CLICK = {"curbhigh", "woodencurb10", "metalbar"}   # thin objects: 2-corner box
CHUNK = 200               # frames per tracker call (GPU memory)
STOP_AFTER = 8            # lost this many frames in a row = left the view
MIN_AREA = 100            # px; smaller masks count as lost

# Corrections after reviewing the filmstrips, per trace "side_trace":
#   "clicks": [frame, ...]        extra clicks (asked in the click round)
#   "not_visible_after": frame    everything after this frame is not visible
#   "not_visible_before": frame   everything before this frame is not visible
FIXES = {
    # "rear_3157": {"not_visible_after": 683},
    # "rear_3130": {"clicks": [600]},
}

OUT_DIR = ARTIFACTS / "gold" / "clicktrack_v1"
(OUT_DIR / "tracks").mkdir(parents=True, exist_ok=True)
CLICKS_JSON = OUT_DIR / "clicks.json"
WORK = Path("/content/cuv") if IN_COLAB else ARTIFACTS / "cuv_work"
WORK.mkdir(parents=True, exist_ok=True)

BUNDLE_ZIP = ARTIFACTS / cfg["bundle"]
bundle_dir = WORK / Path(cfg["bundle"]).stem
if not IN_COLAB and BUNDLE_ZIP.with_suffix("").exists():
    bundle_dir = BUNDLE_ZIP.with_suffix("")
if not (bundle_dir / "frames.csv").exists():
    assert BUNDLE_ZIP.exists(), f"upload {BUNDLE_ZIP.name} to {BUNDLE_ZIP.parent}"
    local_zip = WORK / BUNDLE_ZIP.name
    if not local_zip.exists():
        shutil.copyfile(BUNDLE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)

frames = pd.read_csv(bundle_dir / "frames.csv")
if ALL_TRACES:
    scope = frames
else:
    scope = select_test_items(frames, cfg["split"], step=1)
traces = []
for (side, trace), g in scope.groupby(["side", "trace"]):
    g = final_approach(g)
    click = int(g.iloc[(g["dist_m"] - CLICK_DIST_M).abs().argmin()]["frame_idx"])
    traces.append({"key": f"{side}_{trace}", "side": side, "trace": int(trace),
                   "category": g["label_class"].iloc[0], "object": g["object"].iloc[0],
                   "frames": sorted(frames[(frames["side"] == side) & (frames["trace"] == trace)]["frame_idx"].astype(int)),
                   "click_frame": click})


def frame_path(t, f):
    return bundle_dir / "frames" / t["key"] / f"{int(f):05d}.jpg"


clicks = _json.loads(CLICKS_JSON.read_text()) if CLICKS_JSON.exists() else {}
print(f"{len(traces)} traces, {sum(len(t['frames']) for t in traces)} frames -> {OUT_DIR}")
print(f"clicks so far: {sum(len(v) for v in clicks.values())} on {len(clicks)} traces")

## 1. Click round (resumable; saves after every click)

For each trace you get one frame (~3 m from the object) and click the named
object; thin objects (curbs, bar) take a 2-corner box. Click the object **the
vehicle is approaching**, not a look-alike further away. **Skip** if it is not
visible in that frame. Extra clicks listed in `FIXES` are asked here too.

In [ ]:
todo = []
for t in traces:
    have = {int(f) for f in clicks.get(t["key"], {})}
    wanted = [t["click_frame"]] + [int(f) for f in FIXES.get(t["key"], {}).get("clicks", [])]
    todo += [(t, f) for f in wanted if f not in have]
print(f"clicks to ask: {len(todo)}")
for n, (t, f) in enumerate(todo, 1):
    mode = "box" if t["category"] in BOX_CLICK else "point"
    what = "draw a BOX around" if mode == "box" else "click"
    title = (f"[{n}/{len(todo)}] {t['key']} - {what} the {t['object']} "
             f"(the one the vehicle approaches) - frame {f}")
    ask = ask_click_colab if IN_COLAB else ask_click_local
    res = ask(frame_path(t, f), title, mode)
    clicks.setdefault(t["key"], {})[str(f)] = (
        None if res is None else {("box" if len(res) == 4 else "point"): list(map(float, res))})
    CLICKS_JSON.write_text(_json.dumps(clicks, indent=1))
print("click round done")

## 2. Track (GPU; resumable)

Per click: track forward and backward in chunks of `CHUNK` frames, each chunk
seeded with the last box. Once the object is lost for `STOP_AFTER` frames in a
row it has left the view and stays "not visible" (no re-locking onto a
look-alike). Each frame takes the track of the nearest click; `FIXES`
cut-offs apply last. A trace is re-tracked only when its clicks or fixes change.

In [ ]:
from PIL import Image
from vision_uss_research.labeling.sam3 import Sam3Engine

engine = None


def track_direction(paths, prompt):
    """Boxes for paths[0..] (paths[0] = the click frame), chunked."""
    out, start = [], 0
    while start < len(paths):
        seg = paths[start:start + CHUNK]
        imgs = [Image.open(p).convert("RGB") for p in seg]
        by_pos = engine.track(imgs, 0, prompt)
        boxes = [mask_to_box(by_pos.get(k), MIN_AREA) for k in range(len(seg))]
        out += boxes if start == 0 else boxes[1:]
        if start + CHUNK >= len(paths) or boxes[-1] is None:
            break
        prompt = {"box": list(boxes[-1])}
        start += CHUNK - 1          # overlap one frame to re-seed
    out += [None] * (len(paths) - len(out))
    return stop_after_loss(out, STOP_AFTER)


for t in traces:
    tc = {int(f): v for f, v in clicks.get(t["key"], {}).items() if v}
    fix = FIXES.get(t["key"], {})
    sig = hashlib.sha1(_json.dumps([sorted(tc.items()), fix, CHUNK, STOP_AFTER, MIN_AREA],
                                   sort_keys=True, default=str).encode()).hexdigest()[:10]
    out_csv = OUT_DIR / "tracks" / f"{t['key']}.csv"
    meta = OUT_DIR / "tracks" / f"{t['key']}.json"
    if out_csv.exists() and meta.exists() and _json.loads(meta.read_text()).get("sig") == sig:
        continue
    if not tc:
        print(f"{t['key']}: no click - skipped"); continue
    if engine is None:
        engine = Sam3Engine()
    fr = t["frames"]
    tracks = {}
    for f, prompt in tc.items():
        p = fr.index(f)
        fwd = track_direction([frame_path(t, x) for x in fr[p:]], prompt)
        bwd = track_direction([frame_path(t, x) for x in fr[:p + 1][::-1]], prompt)
        tracks[f] = {**dict(zip(fr[:p + 1][::-1], bwd)), **dict(zip(fr[p:], fwd))}
    boxes = apply_cutoffs(merge_by_nearest_click(fr, tracks),
                          fix.get("not_visible_after"), fix.get("not_visible_before"))
    rows = to_gold_rows(t["side"], t["trace"], t["category"], boxes)
    rows.to_csv(out_csv, index=False)
    meta.write_text(_json.dumps({"sig": sig, "clicks": {str(k): v for k, v in tc.items()}, "fixes": fix}))
    print(f"{t['key']} {t['category']}: {int((rows['status'] == 'box').sum())}/{len(rows)} frames boxed")
print("tracking done")

## 3. Review the filmstrips

Every 15th frame of each trace: green box = tracked, red frame number = not
visible. Look for: box on the wrong object, box drifting off, object visible
but marked not visible (or the reverse). Fix in `FIXES` (config cell) and
rerun cells 1-2; only the changed traces re-track.

In [ ]:
STEP = 15
for t in traces:
    csv = OUT_DIR / "tracks" / f"{t['key']}.csv"
    if not csv.exists():
        continue
    r = pd.read_csv(csv).set_index("frame_idx")
    d = frames[(frames["side"] == t["side"]) & (frames["trace"] == t["trace"])].set_index("frame_idx")["dist_m"]
    shown = t["frames"][::STEP]
    cols = 8
    rows_n = int(np.ceil(len(shown) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 2.4, rows_n * 2.0))
    for ax in np.atleast_1d(axes).flat:
        ax.axis("off")
    for ax, f in zip(np.atleast_1d(axes).flat, shown):
        img = cv2.cvtColor(cv2.imread(str(frame_path(t, f))), cv2.COLOR_BGR2RGB)
        row = r.loc[f]
        ok = row["status"] == "box"
        if ok:
            cv2.rectangle(img, (int(row.x0), int(row.y0)), (int(row.x1), int(row.y1)), (0, 220, 0), 4)
        ax.imshow(img)
        ax.set_title(f"f{f} {d.get(f, float('nan')):.1f}m", fontsize=8, color="black" if ok else "red")
    boxed = (r["status"] == "box").mean()
    fig.suptitle(f"{t['key']} - {t['object']} - {boxed:.0%} of frames boxed", fontsize=11)
    plt.tight_layout(); plt.show()

## 4. Export dense ground truth

In [ ]:
parts = [pd.read_csv(p) for p in sorted((OUT_DIR / "tracks").glob("*.csv"))]
gold = pd.concat(parts, ignore_index=True)
# the rating file holds ONLY the held-out test traces - training traces would
# make the service look better than it is; they go to a separate file
test_keys = {f"{s}_{t}" for s, t in
             select_test_items(frames, cfg["split"], step=1)[["side", "trace"]].drop_duplicates().values}
is_test = (gold["side"] + "_" + gold["trace"].astype(str)).isin(test_keys)
gold[is_test].to_csv(OUT_DIR / "clicktrack_gold.csv", index=False)
if (~is_test).any():
    gold[~is_test].to_csv(OUT_DIR / "clicktrack_train.csv", index=False)
display(gold.assign(split=np.where(is_test, "test", "train"))
        .groupby(["split", "category", "status"]).size().unstack(fill_value=0))
print(f"rating file  : clicktrack_gold.csv  - {int(is_test.sum())} frames, "
      f"{gold[is_test].groupby(['side', 'trace']).ngroups} test traces")
if (~is_test).any():
    print(f"training file: clicktrack_train.csv - {int((~is_test).sum())} frames, "
          f"{gold[~is_test].groupby(['side', 'trace']).ngroups} traces (for retraining, never for rating)")
print("next: download; locally:\n"
      "  uv run --with tabulate python scripts/rate_autolabel.py "
      "--gold-dir <folder with ONLY clicktrack_gold.csv> --name clicktrack_v1")